# 02 — Root Cause Analysis

Notebook 01 gave you a verdict for each of the 20 answers: good or bad. "Bad" is where most evaluation work actually stalls, though -- knowing something's wrong doesn't tell you what to go fix. A wrong answer could mean five completely different things, and each one points a different engineer at a different part of the system.

This notebook is intuition-only, on purpose. No automation yet, no formal metrics yet -- just you, reading a wrong answer, and guessing *where* it broke. You'll revisit these exact guesses in notebook `06`, once you've built the precise vocabulary to check whether your intuition was right.


In [ ]:
import json
import os

DATA_DIR = os.path.join("..", "data")
with open(os.path.join(DATA_DIR, "eval_examples.json")) as f:
    examples = json.load(f)


## The five (or six) ways an answer goes wrong

```
Wrong Answer
  |
  +-- Retrieval failed        -- the wrong document entirely was pulled in
  +-- Chunking failed          -- right document, but the retrieved slice cut off the answer
  +-- Generation hallucinated  -- the answer states something the context never said
  +-- Generation missed it     -- the context had the answer; the model didn't use it
  +-- Should have refused      -- context doesn't cover it; the model guessed instead of declining
  +-- Ground truth is wrong    -- the "gold" label itself is debatable, mislabeled, or overly strict
```

That last one isn't in most textbook versions of this list, and it's tempting to leave it out -- "the benchmark might be wrong" can turn into an excuse for waving away every failure. Keep it anyway, but hold it to a higher bar than the others: only reach for it when you can point at the specific reason the gold label looks wrong, the same way you'd justify any other verdict.


## Worked example 1 — the one that looked like generation, but isn't quite

Back to example 8 from notebook 01, this time asking specifically *where* it broke instead of *whether* it's wrong.


In [ ]:
ex = examples[8]
print("Question:", ex["question"])
print("Context contains gold text?", ex["gold_answer"][:40].lower() in ex["context_text"].lower())
print("Gold:", ex["gold_answer"])
print("System:", ex["system_answer"])


**Root cause:** not retrieval (the right document is here), not chunking (the exact clause is inside the window), not hallucination (the system didn't invent anything, it just declined). That leaves generation-missed-it or should-have-refused, and it's genuinely close between them. The clause never uses the word "cap" -- it's an *exclusion* of indirect damages, not a stated dollar limit -- so from the model's point of view, refusing wasn't obviously wrong; it looked for cap language and didn't find it. Call this one **generation missed context, with a label/task-definition mismatch as the contributing factor**: the category "Cap On Liability" is broader in CUAD's own taxonomy than what the model assumed it meant.


## Worked example 2 — when the "wrong" answer is actually the better one

This is the example every root-cause exercise eventually runs into, and it's worth seeing early rather than being caught off guard by it later.


In [ ]:
ex = examples[13]
print("Question:", ex["question"])
print("Gold:", ex["gold_answer"])
print("System:", ex["system_answer"])


**Root cause:** read the full context for this one yourself (`examples[13]["context_text"]`) before accepting the "no" label at face value. Random House Tower is described as a mixed-use tower housing a publisher's headquarters and apartments; 888 7th Avenue is described as an office skyscraper housing Vornado Realty Trust's headquarters. Both are, in plain English, real estate. The system said "yes" and gave a specific, accurate reason for it. HotpotQA's own label says "no."

This is the ground-truth-is-questionable category, and it's not a cop-out here -- you can point at exactly why: the question as stored is broad ("both used for real estate?") in a way that's true of nearly any building, which suggests the original question this was derived from was more specific, and something was lost in how it got here. **Flag it, note your reasoning, and move on** -- don't silently exclude it, and don't let it convince you every "wrong" answer is secretly right. This is the one that actually is.


## Your turn

Go back through every example you marked "bad" in notebook 01's `my_judgments`. For each, pick one root cause from the list above and write down the specific evidence -- not just the label, the reason, the same way both worked examples did.


In [ ]:
root_causes = {
    8: {"cause": "generation_missed_context", "evidence": "Gold clause is inside the context window; system claimed no info because the clause never says the word 'cap'."},
    13: {"cause": "ground_truth_questionable", "evidence": "Both buildings are described as real estate in the context; system's 'yes' is defensible, gold's 'no' is not clearly supported."},
    # Add every other index you marked "bad" in notebook 01:
    # 6: {"cause": "", "evidence": ""},
    # 9: {"cause": "", "evidence": ""},
}

print(f"Root-caused so far: {len(root_causes)}")


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Root cause | The specific pipeline stage a failure traces back to, not just "it was wrong" |
| Should-have-refused | A failure where the honest answer was "I don't know," and the system guessed instead |
| Ground-truth-questionable | A "wrong" verdict that traces back to a debatable or overly narrow gold label, not the system |

Two things worth carrying forward: first, "wrong" is never a complete diagnosis on its own -- it's the start of a question, not the end of one. Second, you just found a real example where the benchmark's own label was the weak link, not the system being evaluated. That's not a reason to distrust every gold answer going forward; it's a reason to keep asking the same question about the label that you ask about the system: *what's the actual evidence here?*

**Next up:** notebooks `03` through `06` build the precise vocabulary -- retrieval, context relevance, faithfulness, correctness -- to check these intuition-only guesses against something more rigorous than a gut call.

**Exercises:**

- Finish `root_causes` for every "bad" example from notebook 01.
- For any example you tagged `generation_missed_context`, guess what a slightly different question phrasing might have changed. Would rewording "Cap On Liability" as "Does this contract limit or exclude any type of damages?" have changed the system's answer to example 8?
- Try to find a *second* example in the 20 where you suspect the gold label itself might be debatable. What's your specific evidence, not just a hunch?
